In [2]:
import os
import sys
import time
import subprocess
import shutil
from io import StringIO
from typing import List, Optional


import pandas as pd
import numpy as np
from tqdm import tqdm
from Bio import Entrez, SeqIO, AlignIO, Phylo
from Bio.SeqRecord import SeqRecord
import matplotlib.pyplot as plt

In [5]:
ASSOC_CSV = 'CLOVER_0.1_MammalViruses_AssociationsFlatFile.csv' # <- change to your path
OUTDIR = 'clover_bat_cov_output'
ENTREZ_EMAIL = 'princezarzees5075@gmail.com' # <- change to your email
ENTREZ_API_KEY = None # Optional: set your NCBI API key string here if you have one


# thresholds
MIN_HOST_NT_LEN = 800 # minimum nt length for a host mitochondrion marker
MIN_SPIKE_AA_LEN = 650 # minimum aa length for a spike protein
MIN_GENOME_NT_LEN = 20000


# Make directories
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(os.path.join(OUTDIR, 'hosts'), exist_ok=True)
os.makedirs(os.path.join(OUTDIR, 'viruses'), exist_ok=True)
os.makedirs(os.path.join(OUTDIR, 'alignments'), exist_ok=True)
os.makedirs(os.path.join(OUTDIR, 'trees'), exist_ok=True)


# Set Entrez credentials
Entrez.email = ENTREZ_EMAIL
if ENTREZ_API_KEY:
    Entrez.api_key = ENTREZ_API_KEY


# helper
def which(cmd):
    return shutil.which(cmd)


print('mafft:', which('mafft'))
print('iqtree2:', which('iqtree2'))
print('iqtree:', which('iqtree'))

mafft: /Users/PSLabMonisha/miniforge3/envs/snakemake-tutorial/bin/mafft
iqtree2: None
iqtree: /Users/PSLabMonisha/miniforge3/envs/snakemake-tutorial/bin/iqtree


In [8]:
print('Loading associations...')
df = pd.read_csv(ASSOC_CSV, dtype=str,encoding='latin-1')
# normalize column names
df.columns = [c.strip() for c in df.columns]
# make lower-cased HostOrder & VirusFamily for robust filtering
if 'HostOrder' not in df.columns or 'VirusFamily' not in df.columns:
    raise ValueError('Expected columns HostOrder and VirusFamily in associations CSV')


df['HostOrder'] = df['HostOrder'].astype(str).str.lower()
df['VirusFamily'] = df['VirusFamily'].astype(str).str.lower()


df_bat_cov = df[(df['HostOrder'] == 'chiroptera') & (df['VirusFamily'] == 'coronaviridae')].copy()
print(f'Total associations: {len(df)}; bat x coronavirus: {len(df_bat_cov)}')


# write subset for reference
subset_path = os.path.join(OUTDIR, 'clover_bat_cov_subset.csv')
df_bat_cov.to_csv(subset_path, index=False)
print('Wrote filtered subset to', subset_path)

Loading associations...
Total associations: 57406; bat x coronavirus: 55
Wrote filtered subset to clover_bat_cov_output/clover_bat_cov_subset.csv


In [30]:
unique_hosts = sorted(df_bat_cov['Host'].dropna().unique())
print(f'Unique bat hosts ({len(unique_hosts)}):')
for h in unique_hosts[:20]:
    print(' ', h)


# Prefer VirusTaxID column if available
if 'VirusTaxID' in df_bat_cov.columns:
    unique_viruses = sorted(df_bat_cov['VirusTaxID'].dropna().unique())
else:
    unique_viruses = sorted(df_bat_cov['Virus'].dropna().unique())


print(f'Unique viruses ({len(unique_viruses)}):')
for v in unique_viruses[:20]:
    print(' ', v)

Unique bat hosts (22):
  chaerephon plicatus
  hipposideros abae
  hipposideros caffer
  hipposideros crumeniferus
  miniopterus magnater
  miniopterus pusillus
  miniopterus schreibersii
  pipistrellus abramus
  pipistrellus pipistrellus
  rhinolophus affinis
  rhinolophus blasii
  rhinolophus cornutus
  rhinolophus euryale
  rhinolophus ferrumequinum
  rhinolophus macrotis
  rhinolophus mehelyi
  rhinolophus pearsonii
  rhinolophus pusillus
  rhinolophus sinicus
  rousettus leschenaultii
Unique viruses (9):
  11137
  693998
  693999
  694000
  694001
  694006
  694007
  694008
  694009


In [10]:
ENTREZ_SLEEP = 0.34


from Bio import Entrez


def entrez_search_ids(db: str, term: str, retmax: int = 50) -> List[str]:
    try:
        handle = Entrez.esearch(db=db, term=term, retmax=retmax, retmode='xml')
        rec = Entrez.read(handle)
        handle.close()
        time.sleep(ENTREZ_SLEEP)
        return rec.get('IdList', [])
    except Exception as e:
        print('Entrez search failed for', db, term, '->', e)
        return []


def entrez_fetch_fasta(db: str, id_: str, rettype: str = 'fasta', retmode: str = 'text') -> Optional[str]:
    try:
        h = Entrez.efetch(db=db, id=id_, rettype=rettype, retmode=retmode)
        txt = h.read()
        h.close()
        time.sleep(ENTREZ_SLEEP)
        return txt
    except Exception as e:
        print('Entrez fetch failed for', db, id_, '->', e)
        return None


from io import StringIO


def parse_fasta_from_text(txt: str) -> List[SeqRecord]:
    return list(SeqIO.parse(StringIO(txt), 'fasta'))


def sanitize_header(s: str) -> str:
    return s.strip().replace(' ', '_').replace('/', '_').replace('|', '_')

In [11]:
def find_host_mitogenome_or_marker(species_name: str, min_len: int = MIN_HOST_NT_LEN) -> Optional[SeqRecord]:
    """
    Try a sequence of prioritized queries and return the first SeqRecord >= min_len
    """
    queries = [
        f'"{species_name}"[Organism] AND mitochondrion[filter] AND complete genome',
        f'"{species_name}"[Organism] AND mitochondrion[filter] AND (cytochrome b[Title] OR cytb[Title])',
        f'"{species_name}"[Organism] AND (cytochrome b[Title] OR cytb[Title])',
        f'"{species_name}"[Organism] AND (COI OR "cytochrome c oxidase subunit I")[Title]',
        f'"{species_name}"[Organism] AND mitochondrion[filter]'
    ]
    for q in queries:
        ids = entrez_search_ids('nucleotide', q, retmax=10)
        if not ids:
            continue
        for id_ in ids:
            fasta_txt = entrez_fetch_fasta('nucleotide', id_)
            if not fasta_txt:
                continue
            recs = parse_fasta_from_text(fasta_txt)
            if not recs:
                continue
            rec = recs[0]
            if len(rec.seq) >= min_len:
                rec.id = sanitize_header(species_name)
                rec.description = f"{species_name}|ncbi:{id_}"
                return rec
    return None

In [12]:
def find_viral_spike_or_genome(taxid_or_name: str, min_prot_len: int = MIN_SPIKE_AA_LEN, min_gen_len: int = MIN_GENOME_NT_LEN) -> Optional[SeqRecord]:
    """
    If taxid_or_name looks numeric, treat as taxid (txidNNN). Otherwise try to search by virus name.
    Prefer spike protein from 'protein' db; fallback to complete genome from 'nucleotide'.
    """
    is_taxid = False
    try:
        _ = int(str(taxid_or_name))
        is_taxid = True
    except Exception:
        is_taxid = False

    if is_taxid:
        q_prot = f'txid{taxid_or_name}[Organism:exp] AND (spike glycoprotein OR spike protein OR S protein)'
    else:
        q_prot = f'"{taxid_or_name}"[Organism] AND (spike glycoprotein OR spike protein OR S protein)'

    prot_ids = entrez_search_ids('protein', q_prot, retmax=20)
    for pid in prot_ids:
        fasta_txt = entrez_fetch_fasta('protein', pid)
        if fasta_txt:
            recs = parse_fasta_from_text(fasta_txt)
            if recs and len(recs[0].seq) >= min_prot_len:
                rec = recs[0]
                label = f"virus_{taxid_or_name}" if is_taxid else sanitize_header(taxid_or_name)
                rec.id = label
                rec.description = f"spike|ncbi:{pid}"
                return rec

    # fallback: nucleotide complete genome
    if is_taxid:
        q_gen = f'txid{taxid_or_name}[Organism:exp] AND complete genome'
    else:
        q_gen = f'"{taxid_or_name}"[Organism] AND complete genome'

    gen_ids = entrez_search_ids('nucleotide', q_gen, retmax=20)
    for gid in gen_ids:
        fasta_txt = entrez_fetch_fasta('nucleotide', gid)
        if fasta_txt:
            recs = parse_fasta_from_text(fasta_txt)
            if recs and len(recs[0].seq) >= min_gen_len:
                rec = recs[0]
                rec.id = f"virus_{taxid_or_name}"
                rec.description = f"genome|ncbi:{gid}"
                return rec

    return None


In [13]:
print('Fetching host sequences...')
host_records = []
for h in tqdm(unique_hosts, desc='hosts'):
    try:
        rec = find_host_mitogenome_or_marker(h)
        if rec:
            host_records.append(rec)
        else:
            print('No host seq for', h)
    except Exception as e:
        print('Error for host', h, e)

hosts_fasta = os.path.join(OUTDIR, 'hosts', 'hosts_sequences.fasta')
if host_records:
    SeqIO.write(host_records, hosts_fasta, 'fasta')
    print('Wrote hosts fasta to', hosts_fasta)
else:
    print('No host sequences obtained.')

Fetching host sequences...


hosts:  18%|█▊        | 4/22 [00:06<00:30,  1.71s/it]

No host seq for hipposideros crumeniferus


hosts:  59%|█████▉    | 13/22 [00:57<01:38, 10.90s/it]

No host seq for rhinolophus euryale


hosts: 100%|██████████| 22/22 [01:13<00:00,  3.36s/it]

Wrote hosts fasta to clover_bat_cov_output/hosts/hosts_sequences.fasta


In [14]:
print('Fetching viral sequences...')
viral_records = []
for v in tqdm(unique_viruses, desc='viruses'):
    try:
        rec = find_viral_spike_or_genome(v)
        if rec:
            viral_records.append(rec)
        else:
            print('No viral seq for', v)
    except Exception as e:
        print('Error for virus', v, e)

viruses_fasta = os.path.join(OUTDIR, 'viruses', 'viruses_sequences.fasta')
if viral_records:
    SeqIO.write(viral_records, viruses_fasta, 'fasta')
    print('Wrote viruses fasta to', viruses_fasta)
else:
    print('No viral sequences obtained.')

Fetching viral sequences...


viruses: 100%|██████████| 9/9 [00:14<00:00,  1.56s/it]

Wrote viruses fasta to clover_bat_cov_output/viruses/viruses_sequences.fasta


In [15]:
def run_mafft(in_fasta: str, out_fasta: str) -> bool:
    mafft_path = which('mafft')
    if mafft_path is None:
        print('MAFFT not found on PATH. Install mafft or update PATH.')
        return False
    cmd = [mafft_path, '--auto', in_fasta]
    print('Running:', ' '.join(cmd))
    with open(out_fasta, 'w') as outfh:
        subprocess.run(cmd, stdout=outfh, stderr=subprocess.PIPE, check=True)
    print('MAFFT wrote', out_fasta)
    return True

# hosts alignment
host_aln = os.path.join(OUTDIR, 'alignments', 'hosts_aligned.fasta')
if os.path.exists(hosts_fasta) and os.path.getsize(hosts_fasta) > 0:
    try:
        ok = run_mafft(hosts_fasta, host_aln)
    except Exception as e:
        print('MAFFT failed for hosts:', e)
        ok = False
    if not ok:
        # fallback: simple padding (not recommended) — write original file as 'alignment'
        print('Falling back: copying hosts fasta to alignment (no true alignment)')
        shutil.copy(hosts_fasta, host_aln)
else:
    print('No hosts fasta available; skipping host alignment')

# viruses alignment
virus_aln = os.path.join(OUTDIR, 'alignments', 'viruses_aligned.fasta')
if os.path.exists(viruses_fasta) and os.path.getsize(viruses_fasta) > 0:
    try:
        ok = run_mafft(viruses_fasta, virus_aln)
    except Exception as e:
        print('MAFFT failed for viruses:', e)
        ok = False
    if not ok:
        print('Falling back: copying viruses fasta to alignment (no true alignment)')
        shutil.copy(viruses_fasta, virus_aln)
else:
    print('No viruses fasta available; skipping virus alignment')

Running: /Users/PSLabMonisha/miniforge3/envs/snakemake-tutorial/bin/mafft --auto clover_bat_cov_output/hosts/hosts_sequences.fasta
MAFFT wrote clover_bat_cov_output/alignments/hosts_aligned.fasta
Running: /Users/PSLabMonisha/miniforge3/envs/snakemake-tutorial/bin/mafft --auto clover_bat_cov_output/viruses/viruses_sequences.fasta
MAFFT wrote clover_bat_cov_output/alignments/viruses_aligned.fasta


In [22]:
def find_iqtree() -> Optional[str]:
    for cmd in ('iqtree2', 'iqtree'):
        p = which(cmd)
        if p:
            return p
    return None

IQCMD = find_iqtree()
if IQCMD is None:
    raise RuntimeError('IQ-TREE not found. Install iqtree2 or iqtree and ensure it is on your PATH.')
print('IQ-TREE binary detected as:', IQCMD)


def run_iqtree_on_alignment(aln_path: str, is_protein: bool = False, prefix: str = None) -> str:
    aln_abs = os.path.abspath(aln_path)
    workdir = os.path.dirname(aln_abs)
    if prefix is None:
        prefix = os.path.splitext(os.path.basename(aln_abs))[0]

    cmd = [IQCMD, '-s', aln_abs, '-m', 'MFP', '-bb', '1000', '-alrt', '1000', '-nt', 'AUTO', '-pre', prefix]
    if is_protein:
        cmd.extend(['-st', 'AA'])

    print('Running IQ-TREE:', ' '.join(cmd))
    subprocess.run(cmd, cwd=workdir, check=True)

    treefile = os.path.join(workdir, prefix + '.treefile')
    if os.path.exists(treefile):
        return treefile
    alt = os.path.join(workdir, prefix + '.contree')
    if os.path.exists(alt):
        return alt
    raise RuntimeError(f'IQ-TREE did not produce expected treefile for {aln_path}')


def is_protein_fasta(fasta_path: str) -> bool:
    rec = next(SeqIO.parse(fasta_path, 'fasta'))
    alpha = set(str(rec.seq).upper())
    return not alpha.issubset(set('ATCGUN-'))

# Run IQ-TREE for hosts
if os.path.exists(host_aln):
    host_treefile = run_iqtree_on_alignment(host_aln, is_protein=False, prefix='hosts_aln')
    shutil.copy(host_treefile, os.path.join(OUTDIR, 'trees', 'hosts_tree.newick'))

# Run IQ-TREE for viruses
if os.path.exists(virus_aln):
    virus_is_protein = is_protein_fasta(virus_aln)
    virus_treefile = run_iqtree_on_alignment(virus_aln, is_protein=virus_is_protein, prefix='viruses_aln')
    shutil.copy(virus_treefile, os.path.join(OUTDIR, 'trees', 'viruses_tree.newick'))


IQ-TREE binary detected as: /Users/PSLabMonisha/miniforge3/envs/snakemake-tutorial/bin/iqtree
Running IQ-TREE: /Users/PSLabMonisha/miniforge3/envs/snakemake-tutorial/bin/iqtree -s /Users/PSLabMonisha/Desktop/simulation_exps/clover/clover_bat_cov_output/alignments/hosts_aligned.fasta -m MFP -bb 1000 -alrt 1000 -nt AUTO -pre hosts_aln
IQ-TREE version 3.0.1 for MacOS ARM 64-bit built Jul  9 2025
Developed by Bui Quang Minh, Thomas Wong, Nhan Ly-Trong, Huaiyan Ren
Contributed by Lam-Tung Nguyen, Dominik Schrempf, Chris Bielow,
Olga Chernomor, Michael Woodhams, Diep Thi Hoang, Heiko Schmidt

Host:    CS-5095.local (SSE4.2, 24 GB RAM)
Command: /Users/PSLabMonisha/miniforge3/envs/snakemake-tutorial/bin/iqtree -s /Users/PSLabMonisha/Desktop/simulation_exps/clover/clover_bat_cov_output/alignments/hosts_aligned.fasta -m MFP -bb 1000 -alrt 1000 -nt AUTO -pre hosts_aln
Seed:    751207 (Using SPRNG - Scalable Parallel Random Number Generator)
Time:    Mon Sep 22 23:56:56 2025
Kernel:  SSE2 - auto-d

In [23]:
def plot_and_save_tree(treefile: str, out_png: str, title: str = ''):
    try:
        tree = Phylo.read(treefile, 'newick')
        fig = plt.figure(figsize=(8, max(4, len(tree.get_terminals()) * 0.2)))
        ax = fig.add_subplot(1,1,1)
        Phylo.draw(tree, axes=ax, do_show=False)
        if title:
            plt.title(title)
        plt.tight_layout()
        plt.savefig(out_png, dpi=200)
        plt.close(fig)
        print('Saved tree plot to', out_png)
    except Exception as e:
        print('Failed to plot tree', treefile, e)

host_treefile = os.path.join(OUTDIR, 'trees', 'hosts_tree.newick')
virus_treefile = os.path.join(OUTDIR, 'trees', 'viruses_tree.newick')

if os.path.exists(host_treefile):
    plot_and_save_tree(host_treefile, os.path.join(OUTDIR, 'trees', 'hosts_tree.png'), title='Bat hosts phylogeny')
else:
    print('No host tree available to plot')

if os.path.exists(virus_treefile):
    plot_and_save_tree(virus_treefile, os.path.join(OUTDIR, 'trees', 'viruses_tree.png'), title='Coronavirus phylogeny')
else:
    print('No virus tree available to plot')

Saved tree plot to clover_bat_cov_output/trees/hosts_tree.png
Saved tree plot to clover_bat_cov_output/trees/viruses_tree.png


In [24]:
print('\nOutputs are in', OUTDIR)
for root, dirs, files in os.walk(OUTDIR):
    level = root.replace(OUTDIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  - {f}")

print('\nDone. If you want:')
print('- fetch multiple sequences per taxon (e.g., all spike proteins) and build a larger ML tree')
print("- limit hosts to a single marker (e.g., cytb) only")
print("- run IQ-TREE with custom model selection or partitioning")


Outputs are in clover_bat_cov_output
clover_bat_cov_output/
  - clover_bat_cov_subset.csv
  trees/
    - hosts_tree.newick
    - viruses_tree.newick
    - viruses_tree.png
    - hosts_tree.png
  hosts/
    - hosts_sequences.fasta
  alignments/
    - viruses_aligned.fasta
    - hosts_aln.iqtree
    - viruses_aln.splits.nex
    - viruses_aln.bionj
    - hosts_aligned.fasta
    - viruses_aln.log
    - hosts_aln.ckp.gz
    - hosts_aln.mldist
    - hosts_aln.model.gz
    - viruses_aln.ckp.gz
    - viruses_aln.mldist
    - hosts_aln.bionj
    - hosts_aln.contree
    - viruses_aln.model.gz
    - viruses_aln.contree
    - viruses_aln.treefile
    - viruses_aln.iqtree
    - hosts_aln.splits.nex
    - hosts_aln.treefile
    - hosts_aln.log
  viruses/
    - viruses_sequences.fasta

Done. If you want:
- fetch multiple sequences per taxon (e.g., all spike proteins) and build a larger ML tree
- limit hosts to a single marker (e.g., cytb) only
- run IQ-TREE with custom model selection or partitionin

In [34]:
import pandas as pd

# Load associations
df = pd.read_csv('clover_bat_cov_output/clover_bat_cov_subset.csv', dtype=str)
df.columns = [c.strip() for c in df.columns]  # normalize column names
# Format host and virus names
df['Host_fmt'] = df['Host'].str.strip().str.lower().str.replace(' ', '_')
df['Virus_fmt'] = 'virus_' + df['VirusTaxID'].str.strip()

# Create the interaction matrix
interaction_matrix = pd.crosstab(df['Virus_fmt'], df['Host_fmt'])

# Convert to binary (0/1)
interaction_matrix[interaction_matrix > 0] = 1

# Sort rows and columns
interaction_matrix = interaction_matrix.sort_index(axis=0).sort_index(axis=1)

# Save to CSV
interaction_matrix.to_csv('bat_virus_interaction_matrix_formatted.csv')

print('Interaction matrix shape:', interaction_matrix.shape)
print(interaction_matrix.head())

Interaction matrix shape: (9, 22)
Host_fmt      chaerephon_plicatus  hipposideros_abae  hipposideros_caffer  \
Virus_fmt                                                                   
virus_11137                     0                  1                    1   
virus_693998                    0                  0                    0   
virus_693999                    0                  0                    0   
virus_694000                    0                  0                    0   
virus_694001                    0                  0                    0   

Host_fmt      hipposideros_crumeniferus  miniopterus_magnater  \
Virus_fmt                                                       
virus_11137                           1                     0   
virus_693998                          0                     0   
virus_693999                          0                     0   
virus_694000                          0                     1   
virus_694001                        

In [3]:
import pandas as pd

# Load associations
df = pd.read_csv('clover_bat_cov_output/clover_bat_cov_subset.csv', dtype=str)
df.columns = [c.strip() for c in df.columns]  # normalize column names

In [10]:
df['VirusOrder'].value_counts()

VirusOrder
nidovirales    55
Name: count, dtype: int64